# E-Commerce Customer Segmentation

**Customer behaviour analysis and RFM-based segmentation with K-Means**

This project analyses transactional data from a UK-based online retailer and groups customers according to their purchasing behaviour. The objective is to move beyond descriptive analysis and create customer segments that can support retention, targeting and campaign prioritisation.

## Project Objectives

The analysis addresses four business questions:

1. How frequently do customers purchase?
2. Which products are returned most often?
3. Do order values differ between the United Kingdom and international markets?
4. Can customers be grouped into meaningful behavioural segments?

For segmentation, the notebook uses the **RFM framework**:

- **Recency:** days since the customer's latest purchase
- **Frequency:** number of unique completed orders
- **Monetary:** total customer spending

K-Means is applied to scaled and log-transformed RFM features. The number of clusters is evaluated using both the elbow method and the silhouette score.

## 1. Setup and Data Loading

Update `DATA_PATH` if the dataset is stored in a different location.  
`NROWS` is kept at `25_000` to match the original analysis; set it to `None` to use the complete dataset.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from scipy.stats import mannwhitneyu
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
NROWS = 25_000
DATA_PATH = Path(r"C:\Users\User\OneDrive\Υπολογιστής\data\OnlineRetail.csv")

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at: {DATA_PATH}\n"
        "Update DATA_PATH in the setup cell before running the notebook."
    )

df_raw = pd.read_csv(DATA_PATH, nrows=NROWS)
print(f"Raw dataset shape: {df_raw.shape}")
display(df_raw.head())

## 2. Data Quality and Preparation

The preparation stage:

- parses transaction dates,
- removes duplicate records,
- excludes records without a customer identifier from customer-level analysis,
- distinguishes completed sales from cancellations and returns,
- calculates transaction value as `Quantity × UnitPrice`.

In [ ]:
quality_summary = pd.DataFrame({
    "Data type": df_raw.dtypes.astype(str),
    "Missing values": df_raw.isna().sum(),
    "Missing %": (df_raw.isna().mean() * 100).round(2),
    "Unique values": df_raw.nunique()
})

display(quality_summary)
print(f"Duplicate rows: {df_raw.duplicated().sum():,}")

In [ ]:
df = df_raw.copy()

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df = df.drop_duplicates()

# Customer-level analysis requires a valid customer identifier.
df = df.dropna(subset=["CustomerID"]).copy()
df["CustomerID"] = df["CustomerID"].astype("Int64").astype(str)

df["InvoiceNo"] = df["InvoiceNo"].astype(str)
df["IsReturn"] = df["InvoiceNo"].str.startswith("C") | (df["Quantity"] < 0)
df["TotalValue"] = df["Quantity"] * df["UnitPrice"]

sales = df.loc[
    (~df["IsReturn"])
    & (df["Quantity"] > 0)
    & (df["UnitPrice"] > 0)
    & df["InvoiceDate"].notna()
].copy()

returns = df.loc[df["IsReturn"]].copy()

cleaning_summary = pd.Series({
    "Customer-linked transactions": len(df),
    "Completed sales transactions": len(sales),
    "Return/cancellation transactions": len(returns),
    "Unique customers in completed sales": sales["CustomerID"].nunique(),
    "Unique completed invoices": sales["InvoiceNo"].nunique()
}, name="Count")

display(cleaning_summary.to_frame())

## 3. Exploratory Customer and Transaction Analysis

### 3.1 Customer Purchase Frequency

In [ ]:
customer_frequency = (
    sales.groupby("CustomerID")["InvoiceNo"]
    .nunique()
    .sort_values(ascending=False)
)

display(customer_frequency.describe().to_frame("Purchase Frequency"))

plt.figure(figsize=(9, 5))
sns.histplot(customer_frequency, bins=30)
plt.title("Distribution of Customer Purchase Frequency")
plt.xlabel("Number of Unique Orders")
plt.ylabel("Number of Customers")
plt.tight_layout()
plt.show()

The distribution is typically right-skewed: most customers purchase only a small number of times, while a smaller group returns frequently. This pattern supports the use of segmentation rather than treating the customer base as homogeneous.

### 3.2 Most Returned Products

In [ ]:
top_returns = (
    returns.dropna(subset=["Description"])
    .groupby(["StockCode", "Description"], as_index=False)
    .agg(
        ReturnedQuantity=("Quantity", lambda x: abs(x.sum())),
        ReturnTransactions=("InvoiceNo", "nunique")
    )
    .sort_values("ReturnedQuantity", ascending=False)
    .head(10)
)

display(top_returns)

plt.figure(figsize=(10, 6))
sns.barplot(
    data=top_returns,
    x="ReturnedQuantity",
    y="Description"
)
plt.title("Top 10 Products by Returned Quantity")
plt.xlabel("Absolute Returned Quantity")
plt.ylabel("")
plt.tight_layout()
plt.show()

### 3.3 UK versus International Order Values

In [ ]:
invoice_values = (
    sales.groupby(["InvoiceNo", "Country"], as_index=False)
    .agg(OrderValue=("TotalValue", "sum"))
)

invoice_values["Market"] = np.where(
    invoice_values["Country"].eq("United Kingdom"),
    "United Kingdom",
    "Outside United Kingdom"
)

market_summary = (
    invoice_values.groupby("Market")["OrderValue"]
    .agg(["count", "mean", "median", "std"])
    .round(2)
)

display(market_summary)

uk_orders = invoice_values.loc[
    invoice_values["Market"].eq("United Kingdom"), "OrderValue"
]
international_orders = invoice_values.loc[
    invoice_values["Market"].eq("Outside United Kingdom"), "OrderValue"
]

u_statistic, p_value = mannwhitneyu(
    uk_orders,
    international_orders,
    alternative="two-sided"
)

print(f"Mann–Whitney U statistic: {u_statistic:,.0f}")
print(f"p-value: {p_value:.4g}")

plt.figure(figsize=(8, 5))
sns.boxplot(
    data=invoice_values,
    x="Market",
    y="OrderValue",
    showfliers=False
)
plt.title("Completed Order Values by Market")
plt.xlabel("")
plt.ylabel("Order Value")
plt.tight_layout()
plt.show()

The comparison is performed at **invoice level**, rather than by comparing total country quantities. The Mann–Whitney U test evaluates whether the two order-value distributions differ without assuming normality. Statistical significance should be considered together with the group medians and the practical size of the difference.

### 3.4 Highest-Spending Customers

In [ ]:
top_customers = (
    sales.groupby("CustomerID", as_index=False)
    .agg(
        TotalSpent=("TotalValue", "sum"),
        Orders=("InvoiceNo", "nunique"),
        ItemsPurchased=("Quantity", "sum")
    )
    .sort_values("TotalSpent", ascending=False)
    .head(10)
)

display(top_customers)

## 4. RFM Customer Segmentation

The original transaction-level clustering has been upgraded to customer-level RFM segmentation. This ensures that each observation represents one customer and that the clusters describe customer behaviour rather than individual invoices.

In [ ]:
analysis_date = sales["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm = (
    sales.groupby("CustomerID")
    .agg(
        Recency=("InvoiceDate", lambda x: (analysis_date - x.max()).days),
        Frequency=("InvoiceNo", "nunique"),
        Monetary=("TotalValue", "sum")
    )
    .reset_index()
)

# Remove non-positive monetary values if any remain after aggregation.
rfm = rfm.loc[rfm["Monetary"] > 0].copy()

print(f"Customers available for segmentation: {len(rfm):,}")
display(rfm.head())
display(rfm[["Recency", "Frequency", "Monetary"]].describe().round(2))

### 4.1 RFM Distributions

In [ ]:
for feature in ["Recency", "Frequency", "Monetary"]:
    plt.figure(figsize=(8, 4))
    sns.histplot(rfm[feature], bins=30, kde=True)
    plt.title(f"Distribution of {feature}")
    plt.xlabel(feature)
    plt.tight_layout()
    plt.show()

RFM variables are usually strongly right-skewed. A `log1p` transformation reduces the influence of extreme values, while standardisation ensures that no feature dominates K-Means solely because of its measurement scale.

In [ ]:
rfm_features = rfm[["Recency", "Frequency", "Monetary"]].copy()
rfm_log = np.log1p(rfm_features)

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)

rfm_scaled_df = pd.DataFrame(
    rfm_scaled,
    columns=rfm_features.columns,
    index=rfm.index
)

display(rfm_scaled_df.head())

### 4.2 Selecting the Number of Clusters

In [ ]:
model_selection = []

for k in range(2, 9):
    model = KMeans(
        n_clusters=k,
        init="k-means++",
        n_init=20,
        random_state=RANDOM_STATE
    )
    labels = model.fit_predict(rfm_scaled)

    model_selection.append({
        "k": k,
        "Inertia": model.inertia_,
        "Silhouette": silhouette_score(rfm_scaled, labels)
    })

model_selection = pd.DataFrame(model_selection)
display(model_selection.round(4))

plt.figure(figsize=(8, 4))
sns.lineplot(data=model_selection, x="k", y="Inertia", marker="o")
plt.title("Elbow Method")
plt.xlabel("Number of Clusters")
plt.ylabel("Within-Cluster Sum of Squares")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
sns.lineplot(data=model_selection, x="k", y="Silhouette", marker="o")
plt.title("Silhouette Score by Number of Clusters")
plt.xlabel("Number of Clusters")
plt.ylabel("Silhouette Score")
plt.tight_layout()
plt.show()

The elbow plot evaluates how quickly within-cluster variation decreases, while the silhouette score evaluates cluster separation and cohesion. The final choice should combine both diagnostics with business interpretability.

In [ ]:
BEST_K = int(
    model_selection.loc[model_selection["Silhouette"].idxmax(), "k"]
)

print(f"Selected number of clusters based on silhouette score: {BEST_K}")

kmeans = KMeans(
    n_clusters=BEST_K,
    init="k-means++",
    n_init=20,
    random_state=RANDOM_STATE
)

rfm["Cluster"] = kmeans.fit_predict(rfm_scaled)
display(rfm.head())

### 4.3 Two-Dimensional Cluster Visualisation

In [ ]:
pca = PCA(n_components=2)
rfm_pca = pca.fit_transform(rfm_scaled)

pca_df = pd.DataFrame(
    rfm_pca,
    columns=["PC1", "PC2"],
    index=rfm.index
)
pca_df["Cluster"] = rfm["Cluster"].astype(str)

plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="Cluster",
    palette="tab10",
    alpha=0.75
)
plt.title("Customer Segments in PCA Space")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} explained variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} explained variance)")
plt.legend(title="Cluster")
plt.tight_layout()
plt.show()

### 4.4 Cluster Profiles

In [ ]:
cluster_profile = (
    rfm.groupby("Cluster")
    .agg(
        Customers=("CustomerID", "count"),
        MedianRecency=("Recency", "median"),
        MedianFrequency=("Frequency", "median"),
        MedianMonetary=("Monetary", "median"),
        MeanRecency=("Recency", "mean"),
        MeanFrequency=("Frequency", "mean"),
        MeanMonetary=("Monetary", "mean")
    )
)

cluster_profile["CustomerShare_%"] = (
    cluster_profile["Customers"] / cluster_profile["Customers"].sum() * 100
)

display(cluster_profile.round(2).sort_values("MeanMonetary", ascending=False))

In [ ]:
profile_for_heatmap = (
    rfm.groupby("Cluster")[["Recency", "Frequency", "Monetary"]]
    .mean()
)

profile_scaled = pd.DataFrame(
    StandardScaler().fit_transform(profile_for_heatmap),
    index=profile_for_heatmap.index,
    columns=profile_for_heatmap.columns
)

# Lower recency indicates more recent engagement, so reverse its sign
# to make positive heatmap values consistently represent stronger behaviour.
profile_scaled["Recency"] = -profile_scaled["Recency"]

plt.figure(figsize=(8, 4))
sns.heatmap(
    profile_scaled,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    center=0
)
plt.title("Relative RFM Strength by Cluster")
plt.xlabel("Customer Behaviour Dimension")
plt.ylabel("Cluster")
plt.tight_layout()
plt.show()

## 5. Business Interpretation

Use the cluster profile to assign meaningful segment names:

| Behaviour | Possible Segment | Example Action |
|---|---|---|
| Low recency, high frequency, high monetary value | Champions | Retention, exclusivity and loyalty rewards |
| Low recency, moderate frequency and value | Potential Loyalists | Cross-selling and personalised recommendations |
| High recency, historically strong value | At Risk | Reactivation and win-back campaigns |
| High recency, low frequency and low value | Low Engagement | Low-cost automated campaigns |

Segment names should be assigned **after inspecting the calculated cluster profiles**, because numeric cluster labels have no inherent business meaning.

## 6. Conclusion

This analysis combines transaction-level exploration with customer-level RFM segmentation. The upgraded workflow improves the project in three important ways:

- customer segments are based on complete customer behaviour rather than individual invoices,
- the number of clusters is evaluated with both inertia and silhouette score,
- the final output includes interpretable cluster profiles that can support marketing decisions.

The project remains intentionally compact, but now presents a statistically and commercially stronger customer-segmentation workflow.

## Limitations

- The analysis uses the first `25,000` rows when `NROWS=25_000`; results may differ when the full dataset is used.
- K-Means assumes approximately spherical clusters and is sensitive to modelling choices.
- Statistical significance does not necessarily imply commercial significance.
- Campaign recommendations should be validated using future customer behaviour and controlled experiments.